In [1]:
# Calculate total mortality due to PM2.5

In [6]:
import os
import xarray as xr
import numpy as np
from utils.utils import get_scenario_config

In [7]:
# === Health variables ===
# COPD (chronic obstructive pulonary disease)
# LRI (lower respiratory infection)
# IHD (ischemic heart disease)
# DM2 (type 2 diabetes)
# LC (tracheal, bronchus, and lung cancer)
# Stroke
health_vars = ["DM", "LC", "COPD", "LRI"]
age_health_vars = ["Stroke", "IHD"]

In [9]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

MORT_DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/"

ensembles = []
for ens_num in ensemble_members:
    print(f"Processing {scenario} ensemble member {ens_num:02d}")
    all_mort = []
    
    for health_VAR in health_vars:
        dates = f"{years.start}-{years.stop}"
        
        mort_file = f"PM2.5_Mortality_{health_VAR}_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        mort_path = os.path.join(MORT_DIR, mort_file)
        mortality = xr.open_dataarray(mort_path)
        all_mort.append(mortality)

    all_mort_da = xr.concat(all_mort, "health_var")

    total_mortality = all_mort_da.sum("health_var")
    ensembles.append(total_mortality)

total_mort_ens = xr.concat(ensembles, dim=xr.DataArray(ensemble_members, dims="ensemble", name="ensemble"))

description = ("PM2.5 Attributable Mortality for all diseases "
               "and ensemble members - scripts by A.F. Wells (2025)")

total_mort_ens.attrs["description"] = description
total_mort_ens.attrs["scenario"] = scenario
total_mort_ens.attrs["model"] = model

out_file = f"PM2.5_Mortality_total_{model}_{scenario}_{dates}.nc"
out_path = os.path.join(MORT_DIR, out_file)
total_mort_ens.to_netcdf(out_path)

print("All processing complete.")

Processing SSP245_G6 ensemble member 01
Processing SSP245_G6 ensemble member 02
Processing SSP245_G6 ensemble member 03
All processing complete.
